# NLP Subsystem — Training & Model Persistence
### NLP + RL Based Emergency Message Prioritization System (SRS §5)

This notebook covers the **entire NLP pipeline** end to end:

1. Load the training dataset (real dataset attempt → synthetic fallback, SRS §7.1)
2. Exploratory data analysis
3. Preprocessing (SRS §5.1)
4. Category classification — TF-IDF + Logistic Regression (SRS §5.2)
5. Urgency detection — TF-IDF + Logistic Regression (SRS §5.3)
6. **Persist trained models to `models/`** so `app.py` and `rl_agent.py` can load them without retraining
7. Named Entity Recognition + assistance-type keyword matching (SRS §5.4)
8. Duplicate detection (SRS §5.5)
9. Reload check — simulate a fresh process loading the saved artifacts

> ⚠️ **Human-in-the-loop disclosure:** all outputs from this pipeline are decision-support suggestions only. Urgency labels come from a disclosed keyword-tier heuristic, not human-verified ground truth. Location extraction is indicative only, never verified geolocation.

In [ ]:
# This notebook lives at the project root, alongside the `src/` package,
# so it can be run directly from Jupyter (cwd == project root) with no
# sys.path manipulation needed. This cell just makes that assumption
# explicit and fails fast with a clear message if it's not the case.
import sys, os

assert os.path.isdir("src"), (
    "Run this notebook from the project root (the folder containing 'src/', "
    "'app.py', 'rl_agent.py'). If you moved it, update your working directory first."
)

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data import load_dataset
from src.nlp import clean_text, CategoryClassifier, UrgencyClassifier, DuplicateDetector
from src.nlp.preprocessing import preprocessing_backend
from src.nlp.ner import extract_locations, extract_assistance_keywords, ner_backend
from src.config import CATEGORIES, URGENCY_LEVELS, MODELS_DIR

plt.style.use("dark_background")
print("Preprocessing backend:", preprocessing_backend())
print("NER backend:", ner_backend())


## 1. Load the dataset

Attempts a real public disaster-response dataset first; if unavailable (no internet, schema drift, timeout) falls back deterministically to a generated synthetic dataset. Every row is tagged with `provenance` (SRS §7.1).

In [ ]:
df = load_dataset(cache=True)
print(f"Loaded {len(df)} messages")
print(df['provenance'].value_counts())
df.head(10)

## 2. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

cat_counts = df['category'].value_counts().reindex(CATEGORIES)
axes[0].barh(cat_counts.index[::-1], cat_counts.values[::-1], color="#F5A623")
axes[0].set_title("Messages per category")

urg_counts = df['urgency_level'].value_counts().reindex(URGENCY_LEVELS)
axes[1].bar(urg_counts.index, urg_counts.values, color="#2DD4BF")
axes[1].set_title("Messages per urgency level (keyword-tier heuristic label)")

plt.tight_layout()
plt.show()

## 3. Preprocessing (SRS §5.1)

Lowercase → strip URLs/mentions → strip punctuation/digits → lemmatize + stopword removal (spaCy) **or** a documented, functioning fallback (stopword removal only) if spaCy / its model is unavailable.

In [ ]:
samples = df['text'].sample(3, random_state=1).tolist()
for s in samples:
    print("RAW  :", s)
    print("CLEAN:", clean_text(s))
    print()

## 4. Category classification (SRS §5.2)

TF-IDF vectorization (unigrams + bigrams) feeding a Logistic Regression model. Trained on a held-out split; the classification report below is the genuine test-set performance, not training-set performance.

In [ ]:
category_clf = CategoryClassifier()
cat_metrics, cat_report = category_clf.fit(df['text'], df['category'], verbose=False)
print(cat_report)

## 5. Urgency detection (SRS §5.3)

> No public dataset provides ground-truth urgency labels. This project trains on a **disclosed keyword-tier heuristic** label (`urgency_label_source` column) and discloses that provenance everywhere urgency is surfaced in the UI.

In [ ]:
urgency_clf = UrgencyClassifier()
urg_metrics, urg_report = urgency_clf.fit(df['text'], df['urgency_level'], verbose=False)
print(urg_report)

## 6. Persist trained models to disk

Saved with `joblib` so `app.py` and `rl_agent.py` can load them at inference time **without retraining** (SRS §5.2).

In [ ]:
category_clf.save()
urgency_clf.save()

print("Saved artifacts:")
for f in sorted(MODELS_DIR.glob("*.joblib")):
    print(" -", f.name, f"({f.stat().st_size/1024:.1f} KB)")

## 7. Named Entity Recognition + assistance-type keywords (SRS §5.4)

Locations come from a pretrained statistical NER model (spaCy `en_core_web_sm`) when available, else a documented capitalized-phrase fallback heuristic. Assistance-type keywords are **always** produced by a separate, disclosed keyword matcher — never conflated with the learned NER output.

In [ ]:
demo_messages = [
    "Flooding at Riverside Colony, elderly couple trapped on rooftop, water still rising",
    "Need an ambulance urgently at Central Market, someone is unconscious and bleeding",
    "No clean drinking water at Sunrise Township for two days, families are struggling",
]
for msg in demo_messages:
    print("MSG        :", msg)
    print("  locations:", extract_locations(msg))
    print("  assistance:", extract_assistance_keywords(msg))
    print()

## 8. Duplicate detection (SRS §5.5)

TF-IDF cosine similarity against previously seen messages; flags likely duplicates for **human verification** — never auto-discarded.

In [ ]:
detector = DuplicateDetector(threshold=0.8)
seed = df.sample(200, random_state=3)
for _, row in seed.iterrows():
    detector.add(row['message_id'], row['text'])

probe_original = seed.iloc[0]['text']
probe_paraphrase = probe_original + " please respond as soon as possible"
probe_new = "Requesting a status update on the shelter construction timeline at North Bridge"

for probe in [probe_original, probe_paraphrase, probe_new]:
    result = detector.check(probe)
    print(f"PROBE: {probe}")
    print(f"  -> is_duplicate={result.is_duplicate}  similarity={result.similarity:.2f}  best_match_id={result.best_match_id}")
    print()

## 9. Reload check — simulate a fresh process

Confirms the saved artifacts can be loaded **without retraining** and produce predictions, exactly as `app.py` does at inference time.

In [ ]:
fresh_category_clf = CategoryClassifier().load()
fresh_urgency_clf = UrgencyClassifier().load()

test_msg = "Building on fire near Central Market, smoke everywhere, need fire brigade immediately"
category, cat_conf, cat_proba = fresh_category_clf.predict(test_msg)
level, score, urg_conf, urg_proba = fresh_urgency_clf.predict_with_score(test_msg)

print("Message   :", test_msg)
print(f"Category  : {category}  (confidence {cat_conf:.2%})")
print(f"Urgency   : {level}  (score {score:.2f}, confidence {urg_conf:.2%})")
print("Urgency label source:", fresh_urgency_clf.label_source)

## Summary

- ✅ Category classifier and urgency classifier trained on TF-IDF features and persisted to `models/`
- ✅ Reload-without-retrain verified
- ✅ NER + assistance-keyword extraction and duplicate detection demonstrated

**Next:** the Reinforcement Learning subsystem is trained separately in `rl_agent.py` (`python rl_agent.py`), and everything is brought together in the Streamlit demo (`streamlit run app.py`).